In [ ]:
# ============================================================
# Metadata update notebook parameters
# These values will be passed from Fabric pipeline activity
# ============================================================

ingestion_config_id = ""
pipeline_run_id = ""
pipeline_name = ""

run_status = ""   # STARTED / SUCCEEDED / FAILED

source_file_name = ""
source_folder_path = ""
target_path = ""
target_file_name = ""

rows_read = "0"
rows_written = "0"

error_message = ""
new_watermark_value = ""

In [ ]:
from datetime import datetime
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    IntegerType,
    LongType
)

# ============================================================
# Table names
# ============================================================

MD_CONFIG_TABLE = "dbo.md_ingestion_config"
AUDIT_PIPELINE_TABLE = "dbo.audit_pipeline_run"
AUDIT_INGESTION_TABLE = "dbo.audit_ingestion_run"


# ============================================================
# Helper functions
# ============================================================

def safe_string(value):
    """
    Convert None values to blank string.
    """
    if value is None:
        return ""
    return str(value)


def to_int(value, default=0):
    """
    Safely convert value to integer.
    """
    try:
        if value is None or str(value).strip() == "":
            return default
        return int(value)
    except Exception:
        return default


def to_bigint(value, default=0):
    """
    Safely convert value to bigint/integer.
    """
    try:
        if value is None or str(value).strip() == "":
            return default
        return int(value)
    except Exception:
        return default


def validate_inputs():
    """
    Validate mandatory input parameters.
    """

    if safe_string(ingestion_config_id).strip() == "":
        raise Exception("ingestion_config_id is mandatory.")

    if safe_string(pipeline_run_id).strip() == "":
        raise Exception("pipeline_run_id is mandatory.")

    if safe_string(pipeline_name).strip() == "":
        raise Exception("pipeline_name is mandatory.")

    if safe_string(run_status).strip() == "":
        raise Exception("run_status is mandatory.")

    valid_statuses = ["STARTED", "SUCCEEDED", "FAILED"]

    if run_status.upper() not in valid_statuses:
        raise Exception(
            f"Invalid run_status received: {run_status}. "
            f"Allowed values are STARTED, SUCCEEDED, FAILED."
        )


def get_config_row():
    """
    Fetch metadata config row from dbo.md_ingestion_config.
    """

    config_df = spark.sql(f"""
        SELECT
            ingestion_config_id,
            ingestion_name,
            source_type,
            connection_name,
            source_object_name,
            source_folder_path,
            source_file_pattern,
            load_type,
            watermark_column,
            watermark_value,
            target_path,
            target_file_name,
            target_format,
            write_mode,
            schedule_group,
            execution_sequence,
            active_flag,
            created_at,
            updated_at
        FROM {MD_CONFIG_TABLE}
        WHERE ingestion_config_id = '{ingestion_config_id}'
          AND active_flag = true
    """)

    rows = config_df.collect()

    if not rows:
        raise Exception(
            f"No active metadata configuration found for ingestion_config_id: {ingestion_config_id}"
        )

    return rows[0]


def resolve_paths(config):
    """
    Resolve source file name, source path and target path.

    For CourseOfLiCBT, Dataflow Gen2 reads multiple source files.
    Therefore source_file_name can be '*.csv' or a comma-separated list.
    """

    resolved_source_folder_path = safe_string(source_folder_path).strip()
    resolved_source_file_name = safe_string(source_file_name).strip()
    resolved_target_path = safe_string(target_path).strip()
    resolved_target_file_name = safe_string(target_file_name).strip()

    if resolved_source_folder_path == "":
        resolved_source_folder_path = safe_string(config["source_folder_path"]).strip()

    if resolved_source_file_name == "":
        resolved_source_file_name = safe_string(config["source_file_pattern"]).strip()

    if resolved_target_path == "":
        resolved_target_path = safe_string(config["target_path"]).strip()

    if resolved_target_file_name == "":
        resolved_target_file_name = safe_string(config["target_file_name"]).strip()

    resolved_source_folder_path = resolved_source_folder_path.rstrip("/")
    resolved_target_path = resolved_target_path.rstrip("/")

    full_source_path = (
        f"{resolved_source_folder_path}/{resolved_source_file_name}"
        if resolved_source_file_name
        else resolved_source_folder_path
    )

    full_target_path = (
        f"{resolved_target_path}/{resolved_target_file_name}"
        if resolved_target_file_name
        else resolved_target_path
    )

    return (
        resolved_source_file_name,
        full_source_path,
        full_target_path
    )


def get_ingestion_run_id():
    """
    Create deterministic ingestion_run_id so STARTED, SUCCEEDED and FAILED
    update the same ingestion audit row for the same pipeline run.
    """
    return f"{safe_string(pipeline_run_id)}_{safe_string(ingestion_config_id)}"


# ============================================================
# Audit pipeline update
# ============================================================

def update_audit_pipeline_run(config):
    """
    Insert/update dbo.audit_pipeline_run.
    """

    current_time = datetime.now()
    status = run_status.upper()

    schedule_group = safe_string(config["schedule_group"])

    if status == "STARTED":
        row_count_value = 0
        success_count_value = 0
        failure_count_value = 0
        start_time_value = current_time
        end_time_value = None
        error_message_value = ""

    elif status == "SUCCEEDED":
        row_count_value = to_int(rows_written)
        success_count_value = 1
        failure_count_value = 0
        start_time_value = None
        end_time_value = current_time
        error_message_value = ""

    elif status == "FAILED":
        row_count_value = 0
        success_count_value = 0
        failure_count_value = 1
        start_time_value = None
        end_time_value = current_time
        error_message_value = safe_string(error_message)

    else:
        raise Exception(f"Unsupported run_status: {status}")

    schema = StructType([
        StructField("pipeline_run_id", StringType(), False),
        StructField("pipeline_name", StringType(), True),
        StructField("schedule_group", StringType(), True),
        StructField("run_status", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("row_count", IntegerType(), True),
        StructField("success_count", IntegerType(), True),
        StructField("failure_count", IntegerType(), True),
        StructField("error_message", StringType(), True),
        StructField("update_time", TimestampType(), True)
    ])

    data = [(
        safe_string(pipeline_run_id),
        safe_string(pipeline_name),
        schedule_group,
        status,
        start_time_value,
        end_time_value,
        row_count_value,
        success_count_value,
        failure_count_value,
        error_message_value,
        current_time
    )]

    df_update = spark.createDataFrame(data, schema=schema)
    df_update.createOrReplaceTempView("vw_audit_pipeline_update")

    spark.sql(f"""
        MERGE INTO {AUDIT_PIPELINE_TABLE} AS target
        USING vw_audit_pipeline_update AS source
        ON target.pipeline_run_id = source.pipeline_run_id

        WHEN MATCHED THEN UPDATE SET
            target.pipeline_name = source.pipeline_name,
            target.schedule_group = source.schedule_group,
            target.run_status = source.run_status,

            target.start_time =
                CASE
                    WHEN source.run_status = 'STARTED'
                    THEN source.start_time
                    ELSE target.start_time
                END,

            target.end_time =
                CASE
                    WHEN source.run_status IN ('SUCCEEDED', 'FAILED')
                    THEN source.end_time
                    ELSE target.end_time
                END,

            target.row_count =
                CASE
                    WHEN source.run_status = 'SUCCEEDED'
                    THEN source.row_count
                    ELSE target.row_count
                END,

            target.success_count =
                CASE
                    WHEN source.run_status = 'SUCCEEDED'
                    THEN source.success_count
                    ELSE target.success_count
                END,

            target.failure_count =
                CASE
                    WHEN source.run_status = 'FAILED'
                    THEN source.failure_count
                    ELSE target.failure_count
                END,

            target.error_message = source.error_message,
            target.updated_at = source.update_time

        WHEN NOT MATCHED THEN INSERT (
            pipeline_run_id,
            pipeline_name,
            schedule_group,
            run_status,
            start_time,
            end_time,
            row_count,
            success_count,
            failure_count,
            error_message,
            created_at,
            updated_at
        )
        VALUES (
            source.pipeline_run_id,
            source.pipeline_name,
            source.schedule_group,
            source.run_status,
            source.start_time,
            source.end_time,
            source.row_count,
            source.success_count,
            source.failure_count,
            source.error_message,
            source.update_time,
            source.update_time
        )
    """)


# ============================================================
# Audit ingestion update
# ============================================================

def update_audit_ingestion_run(
    config,
    resolved_source_file_name,
    full_source_path,
    full_target_path
):
    """
    Insert/update dbo.audit_ingestion_run.
    """

    current_time = datetime.now()
    status = run_status.upper()

    ingestion_run_id = get_ingestion_run_id()

    previous_watermark_value = safe_string(config["watermark_value"])

    if status == "STARTED":
        start_time_value = current_time
        end_time_value = None
        rows_read_value = 0
        rows_written_value = 0
        error_message_value = ""

    elif status == "SUCCEEDED":
        start_time_value = None
        end_time_value = current_time
        rows_read_value = to_bigint(rows_read)
        rows_written_value = to_bigint(rows_written)
        error_message_value = ""

    elif status == "FAILED":
        start_time_value = None
        end_time_value = current_time
        rows_read_value = to_bigint(rows_read)
        rows_written_value = to_bigint(rows_written)
        error_message_value = safe_string(error_message)

    else:
        raise Exception(f"Unsupported run_status: {status}")

    schema = StructType([
        StructField("ingestion_run_id", StringType(), False),
        StructField("pipeline_run_id", StringType(), True),
        StructField("ingestion_config_id", StringType(), True),

        StructField("source_type", StringType(), True),
        StructField("source_object_name", StringType(), True),
        StructField("source_file_name", StringType(), True),
        StructField("source_path", StringType(), True),
        StructField("target_path", StringType(), True),

        StructField("load_type", StringType(), True),
        StructField("run_status", StringType(), True),

        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_read", LongType(), True),
        StructField("rows_written", LongType(), True),

        StructField("previous_watermark_value", StringType(), True),
        StructField("new_watermark_value", StringType(), True),

        StructField("error_message", StringType(), True),
        StructField("update_time", TimestampType(), True)
    ])

    data = [(
        safe_string(ingestion_run_id),
        safe_string(pipeline_run_id),
        safe_string(ingestion_config_id),

        safe_string(config["source_type"]),
        safe_string(config["source_object_name"]),
        safe_string(resolved_source_file_name),
        safe_string(full_source_path),
        safe_string(full_target_path),

        safe_string(config["load_type"]),
        status,

        start_time_value,
        end_time_value,
        rows_read_value,
        rows_written_value,

        previous_watermark_value,
        safe_string(new_watermark_value),

        error_message_value,
        current_time
    )]

    df_update = spark.createDataFrame(data, schema=schema)
    df_update.createOrReplaceTempView("vw_audit_ingestion_update")

    spark.sql(f"""
        MERGE INTO {AUDIT_INGESTION_TABLE} AS target
        USING vw_audit_ingestion_update AS source
        ON target.ingestion_run_id = source.ingestion_run_id

        WHEN MATCHED THEN UPDATE SET
            target.pipeline_run_id = source.pipeline_run_id,
            target.ingestion_config_id = source.ingestion_config_id,
            target.source_type = source.source_type,
            target.source_object_name = source.source_object_name,
            target.source_file_name = source.source_file_name,
            target.source_path = source.source_path,
            target.target_path = source.target_path,
            target.load_type = source.load_type,
            target.run_status = source.run_status,

            target.start_time =
                CASE
                    WHEN source.run_status = 'STARTED'
                    THEN source.start_time
                    ELSE target.start_time
                END,

            target.end_time =
                CASE
                    WHEN source.run_status IN ('SUCCEEDED', 'FAILED')
                    THEN source.end_time
                    ELSE target.end_time
                END,

            target.rows_read = source.rows_read,
            target.rows_written = source.rows_written,
            target.previous_watermark_value = source.previous_watermark_value,
            target.new_watermark_value = source.new_watermark_value,
            target.error_message = source.error_message,
            target.updated_at = source.update_time

        WHEN NOT MATCHED THEN INSERT (
            ingestion_run_id,
            pipeline_run_id,
            ingestion_config_id,

            source_type,
            source_object_name,
            source_file_name,
            source_path,
            target_path,

            load_type,
            run_status,

            start_time,
            end_time,
            rows_read,
            rows_written,

            previous_watermark_value,
            new_watermark_value,

            error_message,
            created_at,
            updated_at
        )
        VALUES (
            source.ingestion_run_id,
            source.pipeline_run_id,
            source.ingestion_config_id,

            source.source_type,
            source.source_object_name,
            source.source_file_name,
            source.source_path,
            source.target_path,

            source.load_type,
            source.run_status,

            source.start_time,
            source.end_time,
            source.rows_read,
            source.rows_written,

            source.previous_watermark_value,
            source.new_watermark_value,

            source.error_message,
            source.update_time,
            source.update_time
        )
    """)


# ============================================================
# Metadata config update
# ============================================================

def update_md_ingestion_config():
    """
    Update dbo.md_ingestion_config.

    For FULL load:
      - only updated_at is refreshed.

    For INCREMENTAL load:
      - watermark_value is updated only when run_status = SUCCEEDED
        and new_watermark_value is provided.
    """

    current_time = datetime.now()
    status = run_status.upper()

    schema = StructType([
        StructField("ingestion_config_id", StringType(), False),
        StructField("new_watermark_value", StringType(), True),
        StructField("update_time", TimestampType(), True)
    ])

    data = [(
        safe_string(ingestion_config_id),
        safe_string(new_watermark_value),
        current_time
    )]

    df_update = spark.createDataFrame(data, schema=schema)
    df_update.createOrReplaceTempView("vw_md_config_update")

    spark.sql(f"""
        MERGE INTO {MD_CONFIG_TABLE} AS target
        USING vw_md_config_update AS source
        ON target.ingestion_config_id = source.ingestion_config_id

        WHEN MATCHED THEN UPDATE SET
            target.watermark_value =
                CASE
                    WHEN '{status}' = 'SUCCEEDED'
                         AND source.new_watermark_value IS NOT NULL
                         AND source.new_watermark_value <> ''
                    THEN source.new_watermark_value
                    ELSE target.watermark_value
                END,
            target.updated_at = source.update_time
    """)


# ============================================================
# Main execution
# ============================================================

validate_inputs()

run_status = run_status.upper()

config = get_config_row()

resolved_source_file_name, full_source_path, full_target_path = resolve_paths(config)

update_audit_pipeline_run(config)
update_audit_ingestion_run(
    config,
    resolved_source_file_name,
    full_source_path,
    full_target_path
)
update_md_ingestion_config()

print("Metadata update completed successfully.")
print(f"ingestion_config_id : {ingestion_config_id}")
print(f"pipeline_run_id     : {pipeline_run_id}")
print(f"pipeline_name       : {pipeline_name}")
print(f"run_status          : {run_status}")
print(f"source_file_name    : {resolved_source_file_name}")
print(f"source_path         : {full_source_path}")
print(f"target_path         : {full_target_path}")
print(f"rows_read           : {rows_read}")
print(f"rows_written        : {rows_written}")